# Importation des donnees

In [1]:
# importer les jeux de donnees venant de SQL Server

import importlib
import etl.config as cfg
importlib.reload(cfg)

conn = cfg.get_source_engine()

# importer les fonctions de transformation
#from etl.transform import extract_transform_charges_telephoniques, extract_transform_charges_impression

In [2]:
# importer la bibliotheque pandas pour la manipulation des donnees
import pandas as pd

# Tester l'affichage des ChargesTelephoniques

ChargesTelephoniques = pd.read_sql("SELECT * FROM ChargesTelephoniques", conn)
print(ChargesTelephoniques.head(10))

   TelephoniqueID NomDepartement DateOperation      NomRole CodeEmployee  \
0               1   PRODUCTION A       2022-01      MANAGER       YAZ001   
1               2   PRODUCTION A       2022-01  SPECIALISTE       YAZ002   
2               3   PRODUCTION A       2022-01   TECHNICIEN       YAZ003   
3               4   PRODUCTION A       2022-01   TECHNICIEN       YAZ004   
4               5   PRODUCTION B       2022-01      MANAGER       YAZ005   
5               6   PRODUCTION B       2022-01  SUPERVISEUR       YAZ006   
6               7   PRODUCTION B       2022-01  SPECIALISTE       YAZ007   
7               8   PRODUCTION B       2022-01   TECHNICIEN       YAZ008   
8               9   PRODUCTION B       2022-01   TECHNICIEN       YAZ009   
9              10   PRODUCTION B       2022-01  LINE LEADER       YAZ010   

  NumeroTelephone  ForfaitTND  
0        20112233       100.0  
1        50223344        40.0  
2        90334455        20.0  
3        20445566        20.0  
4  

In [3]:
# Tester l'affichage des ChargesImpression

ChargesImpression = pd.read_sql("SELECT * FROM ChargesImpression", conn)
print(ChargesImpression.head())

   ImpressionID NomDepartement DateImpression TypeImpression  NbPages  \
0             1          COSEE     2022-01-01          A4-NB       43   
1             2      DIRECTION     2022-01-01          A4-NB        2   
2             3            EHS     2022-01-01          A4-NB        9   
3             4            EHS     2022-01-01          A4-NB       58   
4             5     ENGENIERIE     2022-01-01          A4-NB       26   

   CoutUnitaire  
0         0.024  
1         0.024  
2         0.024  
3         0.024  
4         0.024  


---
#  Exploration - ChargesTelephoniques

In [8]:
# extract et transform des ChargesImpression et telephoniques
ChargesTelephoniques = extract_transform_charges_telephoniques()
ChargesImpression = extract_transform_charges_impression()

[PIPELINE] Extraction + Transformation ChargesTelephoniques...
[INFO] Validation dates ChargesTelephoniques - colonne 'DateOperation'...
  ✓ Toutes les dates valides (6288 lignes)
[INFO] Normalisation NomDepartement (ChargesTelephoniques)...
  ⚠️  774 département(s) invalide(s) détecté(s)
  ✓ 54 ligne(s) imputée(s) via CodeEmployee
  ⚠️  720 ligne(s) → 'AUTRES'
[INFO] Uniformisation NomDepartement (le plus récurrent par CodeEmployee)...
  ✓ 182 CodeEmployee(s) ont un NomDepartement unique
[INFO] Normalisation CodeEmployee (format: YAZ+nombre ou AUTRES)...
  ✓ CodeEmployee reformatés
[INFO] Propagation NomRole et NumeroTelephone (dernier jour du mois → tous les jours du mois)...
  ✓ 6288 combinaison(s) traité(e)s
[INFO] Suppression doublons ChargesTelephoniques...
  ✓ Aucun doublon détecté
[INFO] Réassignation IDs et tri ChargesTelephoniques...
  ✓ 6288 lignes conservées, IDs réassignés (1-6288)
    - 6288 lignes valides
    - 0 lignes avec date invalide
[PIPELINE] Extraction + Transfor

In [4]:
# afficher les dimensions de ChargesTelephoniques

print(f"Dimensions : {ChargesTelephoniques.shape[0]} lignes x {ChargesTelephoniques.shape[1]} colonnes")

Dimensions : 5726 lignes x 7 colonnes


In [5]:
# Analyse statistique des colonnes ForfaitTND

ChargesTelephoniques[['ForfaitTND']].describe()


,ForfaitTND
count,5608.000000
mean,44.743224
std,26.262420
min,20.000000
25%,20.000000
50%,40.000000
75%,60.000000
max,100.000000


In [6]:
# afficher les informations sur les types de données et les valeurs manquantes de ChargesTelephoniques
ChargesTelephoniques.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5726 entries, 0 to 5725
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TelephoniqueID   5726 non-null   int64  
 1   NomDepartement   5679 non-null   object 
 2   DateOperation    5726 non-null   object 
 3   NomRole          5602 non-null   object 
 4   CodeEmployee     5726 non-null   object 
 5   NumeroTelephone  5726 non-null   object 
 6   ForfaitTND       5608 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 313.3+ KB


In [7]:
# afficher le nombre de valeurs manquantes par colonne dans ChargesTelephoniques
ChargesTelephoniques.isnull().sum()

TelephoniqueID       0
NomDepartement      47
DateOperation        0
NomRole            124
CodeEmployee         0
NumeroTelephone      0
ForfaitTND         118
dtype: int64

In [8]:
# afficher les valeurs uniques de la colonne NomDepartement dans ChargesTelephoniques
ChargesTelephoniques['NomDepartement'].unique()

array(['PRODUCTION A', 'PRODUCTION B', 'DIRECTION', 'EHS', 'ENGENIERIE',
       'IT', 'FINANCE', 'QUALITE', 'ACHAT', 'COSEE', '  ENGENIERIE',
       'LOGISTIQUE', 'TD', 'RH', '  RH', 'NYS', 'OLS', 'PLPP',
       'FINANCE  ', 'DIRECTION  ', '  DIRECTION', None, '  OLS',
       '  QUALITE', '  PRODUCTION B', 'PRODUCTION B  ', '  COSEE',
       'PLPP  ', '  TD', 'RH  ', '  LOGISTIQUE', '  PRODUCTION A',
       '  NYS', 'COSEE  ', '  FINANCE', 'TD  ', '  ACHAT', 'EHS  ',
       'QUALITE  ', 'NYS  ', '  PLPP', 'OLS  ', '  EHS', 'LOGISTIQUE  ',
       '  IT', 'PRODUCTION A  ', 'ACHAT  '], dtype=object)

In [14]:
# afficher les valeurs uniques de la colonne DateOperation dans ChargesTelephoniques
ChargesTelephoniques['DateOperation'].unique()

<DatetimeArray>
['2022-01-01 00:00:00', '2022-02-01 00:00:00', '2022-03-01 00:00:00',
 '2022-04-01 00:00:00', '2022-05-01 00:00:00', '2022-06-01 00:00:00',
 '2022-07-01 00:00:00', '2022-08-01 00:00:00', '2022-09-01 00:00:00',
 '2022-10-01 00:00:00', '2022-11-01 00:00:00', '2022-12-01 00:00:00',
 '2023-01-01 00:00:00', '2023-02-01 00:00:00', '2023-03-01 00:00:00',
 '2023-04-01 00:00:00', '2023-05-01 00:00:00', '2023-06-01 00:00:00',
 '2023-07-01 00:00:00', '2023-08-01 00:00:00', '2023-09-01 00:00:00',
 '2023-10-01 00:00:00', '2023-11-01 00:00:00', '2023-12-01 00:00:00',
 '2024-01-01 00:00:00', '2024-02-01 00:00:00', '2024-03-01 00:00:00',
 '2024-04-01 00:00:00', '2024-05-01 00:00:00', '2024-06-01 00:00:00',
 '2024-07-01 00:00:00', '2024-08-01 00:00:00', '2024-09-01 00:00:00',
 '2024-10-01 00:00:00', '2024-11-01 00:00:00', '2024-12-01 00:00:00',
 '2025-01-01 00:00:00', '2025-02-01 00:00:00', '2025-03-01 00:00:00',
 '2025-04-01 00:00:00', '2025-05-01 00:00:00', '2025-06-01 00:00:00',
 '20

In [15]:
#afficher les valeurs uniques de la colonne NomRole dans ChargesTelephoniques
ChargesTelephoniques['NomRole'].unique()

<StringArray>
[                     'MANAGER',                   'TECHNICIEN',
                  'SPECIALISTE',                  'SUPERVISEUR',
 'SUPERVISEUR COMITE DIRECTION',     'CENTRAL FUNCTION MANAGER',
                'ASSISTANTE DG',                         'HEAD',
                  'LINE LEADER',             'CENTRAL FUNCTION',
                 'TEAM MANAGER']
Length: 11, dtype: string

In [16]:
# afficher les valeurs uniques de la colonne CodeEmployee dans ChargesTelephoniques
ChargesTelephoniques['CodeEmployee'].unique()

array(['YAZ001', 'YAZ097', 'YAZ096', 'YAZ095', 'YAZ094', 'YAZ093',
       'YAZ092', 'YAZ091', 'YAZ090', 'YAZ089', 'YAZ088', 'YAZ087',
       'YAZ086', 'YAZ085', 'YAZ084', 'YAZ083', 'YAZ082', 'YAZ081',
       'YAZ080', 'YAZ079', 'YAZ078', 'YAZ077', 'YAZ076', 'YAZ075',
       'YAZ074', 'YAZ073', 'YAZ072', 'YAZ071', 'YAZ070', 'YAZ069',
       'YAZ099', 'YAZ068', 'YAZ100', 'YAZ102', 'YAZ131', 'YAZ130',
       'YAZ129', 'YAZ128', 'YAZ127', 'YAZ126', 'YAZ125', 'YAZ124',
       'YAZ123', 'YAZ122', 'YAZ121', 'YAZ120', 'YAZ119', 'YAZ118',
       'YAZ117', 'YAZ116', 'YAZ115', 'YAZ114', 'YAZ113', 'YAZ112',
       'YAZ111', 'YAZ110', 'YAZ109', 'YAZ108', 'YAZ107', 'YAZ106',
       'YAZ105', 'YAZ104', 'YAZ103', 'YAZ101', 'YAZ067', 'YAZ098',
       'YAZ065', 'YAZ031', 'YAZ030', 'YAZ029', 'YAZ028', 'YAZ027',
       'YAZ026', 'YAZ025', 'YAZ024', 'YAZ023', 'YAZ022', 'YAZ021',
       'YAZ020', 'YAZ018', 'YAZ017', 'YAZ032', 'YAZ016', 'YAZ014',
       'YAZ013', 'YAZ012', 'YAZ011', 'YAZ010', 'YAZ009', 'YAZ0

In [17]:
# le nombre de numéros de téléphone uniques dans ChargesTelephoniques
print("Nombre de numéros de téléphone uniques :")
print(len(ChargesTelephoniques['NumeroTelephone'].unique()))

Nombre de numéros de téléphone uniques :
131


In [18]:
# le nombre de TelephoniqueID uniques dans ChargesTelephoniques
print("Nombre de TelephoniqueID uniques :")
print(len(ChargesTelephoniques['TelephoniqueID'].unique()))

Nombre de TelephoniqueID uniques :
6288


In [19]:
# afficher les valeurs uniques de la colonne ForfaitTND dans ChargesTelephoniques
ChargesTelephoniques['ForfaitTND'].unique()

array([100,  25,  40,  50,  70,  80,   0,  20], dtype=int64)

In [20]:
# afficher le nombre de doublons dans ChargesTelephoniques (toutes colonnes identiques)
print(f"Doublons (toutes colonnes)   : {ChargesTelephoniques.duplicated().sum()}")

# afficher le nombre de doublons en excluant TelephoniquesID (vraies doublons de données)
cols_sans_id = [col for col in ChargesTelephoniques.columns if col != 'TelephoniqueID']
doublons_sans_id = ChargesTelephoniques.duplicated(subset=cols_sans_id).sum()
print(f"Doublons (sans TelephoniqueID) : {doublons_sans_id}")

Doublons (toutes colonnes)   : 0
Doublons (sans TelephoniqueID) : 0


In [21]:
# afficher les ForfaitTND par NomRole

ChargesTelephoniques.groupby('NomRole')['ForfaitTND'].unique()

NomRole
ASSISTANTE DG                    [80]
CENTRAL FUNCTION                 [50]
CENTRAL FUNCTION MANAGER        [100]
HEAD                              [0]
LINE LEADER                      [20]
MANAGER                         [100]
SPECIALISTE                      [40]
SUPERVISEUR                      [50]
SUPERVISEUR COMITE DIRECTION     [70]
TEAM MANAGER                     [80]
TECHNICIEN                       [25]
Name: ForfaitTND, dtype: object

In [22]:
# Détecter les employés ayant plus d'un numéro de téléphone distinct
phone_counts = ChargesTelephoniques.groupby(['CodeEmployee', 'NumeroTelephone']).size().reset_index(name='NbOccurrences')
multi_phone = phone_counts.groupby('CodeEmployee').filter(lambda x: len(x) > 1)

print(f"Employés avec plusieurs numéros distincts : {multi_phone['CodeEmployee'].nunique()}\n")
multi_phone.sort_values(['CodeEmployee', 'NbOccurrences'], ascending=[True, False])

Employés avec plusieurs numéros distincts : 0



,CodeEmployee,NumeroTelephone,NbOccurrences


In [23]:
# Détecter les employés affectés à plus d'un département
dept_counts = ChargesTelephoniques.groupby(['CodeEmployee', 'NomDepartement']).size().reset_index(name='NbOccurrences')
multi_dept = dept_counts.groupby('CodeEmployee').filter(lambda x: len(x) > 1)

print(f"Employés avec plusieurs départements distincts : {multi_dept['CodeEmployee'].nunique()}\n")
multi_dept.sort_values(['CodeEmployee', 'NbOccurrences'], ascending=[True, False])

Employés avec plusieurs départements distincts : 0



,CodeEmployee,NomDepartement,NbOccurrences


---
#  Exploration - ChargesImpression

In [24]:
# afficher les dimensions de ChargesImpression
print(f"Dimensions : {ChargesImpression.shape[0]} lignes x {ChargesImpression.shape[1]} colonnes")

Dimensions : 15058 lignes x 9 colonnes


In [25]:
# analyse statistique des colonnes NbPages et CoutUnitaire
ChargesImpression[['NbPages', 'CoutUnitaire']].describe()

,NbPages,CoutUnitaire
count,15058.000000,15058.000000
mean,14.817373,0.150606
std,14.836618,0.238111
min,0.000000,0.024000
25%,4.000000,0.024000
50%,8.000000,0.024000
75%,24.000000,0.048000
max,100.000000,0.986000


In [26]:
# afficher les informations sur les types de données et les valeurs manquantes de ChargesImpression
ChargesImpression.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15058 entries, 0 to 15057
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ImpressionID       15058 non-null  int64         
 1   NomDepartement     15058 non-null  string        
 2   DateImpression     15058 non-null  datetime64[ns]
 3   TypeImpression     15058 non-null  string        
 4   NbPages            15058 non-null  int32         
 5   CoutUnitaire       15058 non-null  float64       
 6   FormatPapier       15058 non-null  object        
 7   CouleurImpression  15058 non-null  object        
 8   DateValid          15058 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(1), int32(1), int64(1), object(2), string(2)
memory usage: 897.1+ KB


In [27]:
# afficher le nombre de valeurs manquantes par colonne dans ChargesImpression
ChargesImpression.isnull().sum()

ImpressionID         0
NomDepartement       0
DateImpression       0
TypeImpression       0
NbPages              0
CoutUnitaire         0
FormatPapier         0
CouleurImpression    0
DateValid            0
dtype: int64

In [28]:
# afficher le nombre de doublons dans ChargesImpression (toutes colonnes identiques)
print(f"Doublons (toutes colonnes)   : {ChargesImpression.duplicated().sum()}")

# afficher le nombre de doublons en excluant ImpressionID (vraies doublons de données)
cols_sans_id = [col for col in ChargesImpression.columns if col != 'ImpressionID']
doublons_sans_id = ChargesImpression.duplicated(subset=cols_sans_id).sum()
print(f"Doublons (sans ImpressionID) : {doublons_sans_id}")

Doublons (toutes colonnes)   : 0
Doublons (sans ImpressionID) : 0


In [29]:
# le nombre de numéros d'impression uniques dans ChargesImpression
print("Nombre de numéros d'impression uniques :")
print(len(ChargesImpression['ImpressionID'].unique()))

Nombre de numéros d'impression uniques :
15058


In [30]:
# afficher les valeurs uniques de la colonne NomDepartement dans ChargesImpression
ChargesImpression['NomDepartement'].unique()

<StringArray>
[       'COSEE', 'PRODUCTION B',          'OLS',          'NYS',
   'LOGISTIQUE',       'AUTRES',          'EHS',    'DIRECTION',
           'TD',           'RH',      'QUALITE', 'PRODUCTION A',
           'IT',         'PLPP',      'FINANCE',        'ACHAT']
Length: 16, dtype: string

In [31]:
# afficher les valeurs uniques de la colonne CodeDetailImpression dans ChargesImpression
ChargesImpression['TypeImpression'].unique()

<StringArray>
['A4-NB', 'A4-COULEUR', 'A3-NB', 'A3-COULEUR']
Length: 4, dtype: string

In [32]:
# afficher les valeurs uniques de la colonne NbPages dans ChargesImpression
ChargesImpression['NbPages'].unique()

array([ 43,  39,  33,  47,   4,  20,   2,   5,  26,  58,   9,   6,  40,
        13,  28,  44,   3,  10,  12,  48,  50,   7,  34,  21,  35,  25,
         8,   1,   0,  45,  15,  30,  41,  19,  24,  42,  27,  49,  46,
        18,  16,  29,  38,  23,  14,  37,  11,  22,  31,  32,  17,  36,
        74,  69,  87,  54,  93,  71,  99,  90,  78,  61,  86,  92,  83,
        84,  68,  82,  51,  96,  89,  60,  81,  52, 100,  88,  53,  62,
        80,  72,  59,  79,  75,  65,  57,  94,  64,  56,  76])

In [33]:
# afficher les valeurs uniques de la colonne CoutUnitaire dans ChargesImpression
ChargesImpression['CoutUnitaire'].unique()

array([0.024, 0.493, 0.048, 0.986])

In [34]:
# coherence TypeImpression / CoutUnitaire
detail_info = (
    ChargesImpression
    .groupby('TypeImpression')
    .agg(
        CoutUnitaire=('CoutUnitaire', 'unique'),
        NbCouts=('CoutUnitaire', 'nunique'),
    )
)
# Signaler les incohérences
incoherents_detail_cout = detail_info[detail_info['NbCouts'] > 1]
print(f"Types d'impression incohérents : {len(incoherents_detail_cout)}\n")
detail_info

Types d'impression incohérents : 0



,CoutUnitaire,NbCouts
TypeImpression,,
A3-COULEUR,[0.986],1
A3-NB,[0.048],1
A4-COULEUR,[0.493],1
A4-NB,[0.024],1


In [35]:
ChargesImpression['NbPages'][ChargesImpression['NbPages'] < 0].unique()

array([], dtype=int32)

In [36]:
ChargesImpression['FormatPapier'].unique()

array(['A4', 'A3'], dtype=object)

In [37]:
ChargesImpression['CouleurImpression'].unique()

array(['NOIR ET BLANC', 'COULEUR'], dtype=object)